# Logical Operations on Images

This notebook demonstrates logical AND, OR, and NOT on binary images derived from `selfie.jpg`.
A binary thresholded version of the image is combined with an ellipse mask to make the logical effects easy to observe.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

plt.style.use('seaborn-v0_8-whitegrid')

: 

In [ ]:
def load_image_unicode(image_path, flags=cv2.IMREAD_COLOR):
    image_bytes = np.fromfile(str(image_path), dtype=np.uint8)
    image = cv2.imdecode(image_bytes, flags)
    if image is None:
        raise ValueError(f'Failed to load image: {image_path}')
    return image


image_path = Path('selfie.jpg')
results_dir = Path('results')
results_dir.mkdir(exist_ok=True)

image_bgr = load_image_unicode(image_path)
image_gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)

_, binary_image = cv2.threshold(image_gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

mask = np.zeros_like(binary_image)
center = (binary_image.shape[1] // 2, binary_image.shape[0] // 2)
axes = (binary_image.shape[1] // 4, binary_image.shape[0] // 3)
cv2.ellipse(mask, center, axes, 0, 0, 360, 255, -1)

In [ ]:
and_result = cv2.bitwise_and(binary_image, mask)
or_result = cv2.bitwise_or(binary_image, mask)
not_result = cv2.bitwise_not(binary_image)

images = {
    'Original Grayscale': image_gray,
    'Binary Image (A)': binary_image,
    'Ellipse Mask (B)': mask,
    'A AND B': and_result,
    'A OR B': or_result,
    'NOT A': not_result,
}

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
for ax, (title, img) in zip(axes.ravel(), images.items()):
    cmap = 'gray' if img.ndim == 2 else None
    ax.imshow(img, cmap=cmap, vmin=0, vmax=255)
    ax.set_title(title)
    ax.axis('off')

plt.tight_layout()
plt.show()

fig.savefig(results_dir / '3_logical_operations.png', dpi=180, bbox_inches='tight')

In [ ]:
def white_ratio(image):
    return np.count_nonzero(image == 255) / image.size


for name in ['Binary Image (A)', 'Ellipse Mask (B)', 'A AND B', 'A OR B', 'NOT A']:
    print(f"{name:<18} white-pixel ratio = {white_ratio(images[name]):.4f}")